# Análisis de noticias de jugadores con ESPN + OpenAI

Decidimos usar la API no oficial de ESPN (`site.api.espn.com`) para obtener noticias de las 5 grandes ligas, y la API de OpenAI para clasificarlas y ligarlas a jugadores.

**Limitaciones ya verificadas antes de construir esto (ver bitácora del proyecto):**
- ESPN solo entrega una ventana de noticias **recientes** (~50 artículos más nuevos, cubriendo unos 3-4 días). No hay filtro por fecha ni paginación real (`page` no funciona) ni acceso a noticias históricas.
- Por lo tanto, este notebook produce una **fotografía del presente**, no una serie histórica. Si se quiere construir historial, hay que correr esta captura periódicamente y acumular resultados.
- Se necesita `OPENAI_API_KEY` en `../.env` (no se sube al repo, está en `.gitignore`).

In [1]:
import json
import os

import pandas as pd
import requests
from dotenv import load_dotenv
from openai import OpenAI
from unidecode import unidecode

In [2]:
load_dotenv("../.env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

LIGAS = {
    "Premier League": "eng.1",
    "La Liga": "esp.1",
    "Serie A": "ita.1",
    "Bundesliga": "ger.1",
    "Ligue 1": "fra.1",
}

## 1. Descarga de noticias por liga (ESPN)

Endpoint: `https://site.api.espn.com/apis/site/v2/sports/soccer/{liga}/news`. Pedimos el máximo de artículos (`limit=50`, el tope real del endpoint) para cada una de las 5 ligas y deduplicamos por `id` (una misma noticia puede aparecer listada en más de una liga, p. ej. partidos de Champions).

In [3]:
def fetch_liga_news(slug, limit=50):
    url = f"https://site.api.espn.com/apis/site/v2/sports/soccer/{slug}/news"
    r = requests.get(url, params={"limit": limit}, timeout=15)
    r.raise_for_status()
    return r.json().get("articles", [])

In [4]:
articulos = []
for liga, slug in LIGAS.items():
    for art in fetch_liga_news(slug):
        art["liga"] = liga
        articulos.append(art)

df_news = pd.DataFrame(articulos).drop_duplicates(subset="id").reset_index(drop=True)
print("Total de artículos (tras deduplicar):", len(df_news))
df_news["liga"].value_counts()

Total de artículos (tras deduplicar): 202


liga
Premier League    50
La Liga           46
Serie A           46
Bundesliga        32
Ligue 1           28
Name: count, dtype: int64

## 2. Jugadores ya etiquetados por ESPN

Antes de pedirle a un LLM que adivine qué jugador menciona cada noticia, revisamos el campo `categories` de cada artículo: ESPN ya trae etiquetas estructuradas de tipo `"athlete"` con `athleteId` y nombre. Esto es más confiable que dejar que el LLM extraiga nombres en texto libre (evita errores de ortografía/alucinación), así que lo usamos como fuente de verdad para "qué jugador aparece en esta noticia" y dejamos que OpenAI se encargue de lo que sí requiere lenguaje: categoría, sentimiento y resumen.

No todas las noticias tienen jugador etiquetado (hay notas de equipo, de liga en general, etc.) — esas quedan fuera del cruce por jugador pero se siguen contando para la clasificación general.

In [5]:
def extraer_atletas(categorias):
    return [
        {"athlete_id": c.get("athleteId"), "athlete_name": c.get("description")}
        for c in categorias or []
        if c.get("type") == "athlete"
    ]

df_news["atletas"] = df_news["categories"].apply(extraer_atletas)

filas = []
for _, row in df_news.iterrows():
    for atleta in row["atletas"]:
        filas.append({
            "article_id": row["id"],
            "liga": row["liga"],
            "published": row["published"],
            "headline": row["headline"],
            "athlete_id": atleta["athlete_id"],
            "athlete_name": atleta["athlete_name"],
        })

df_articulo_jugador = pd.DataFrame(filas)
print("Artículos con al menos un jugador etiquetado:", df_news["atletas"].apply(len).gt(0).sum(), "/", len(df_news))
print("Jugadores únicos mencionados:", df_articulo_jugador["athlete_name"].nunique())
df_articulo_jugador.head()

Artículos con al menos un jugador etiquetado: 127 / 202
Jugadores únicos mencionados: 149


,article_id,liga,published,headline,athlete_id,athlete_name
0,49905021,Premier League,2026-09-11T16:46:26Z,Monaco's Lamine Camara says he has 'turned the...,38721,Lamine Camara
1,49905024,Premier League,2026-09-11T16:47:20Z,Tottenham's Roberto De Zerbi says Mykhailo Mud...,299911,Mykhailo Mudryk
2,49904487,Premier League,2026-09-11T18:35:59Z,Arsenal reach agreement to sign 17-year-old wi...,420882,Andria Bartishvili
3,49904189,Premier League,2026-09-11T15:04:37Z,Xabi Alonso enjoys Morgan Rogers and Cole Palm...,265653,Morgan Rogers
4,49904189,Premier League,2026-09-11T15:04:37Z,Xabi Alonso enjoys Morgan Rogers and Cole Palm...,296395,Cole Palmer


## 3. Clasificación con OpenAI (categoría, sentimiento, resumen)

Un llamado por artículo (no por jugador, para no pagar de más) con `response_format` de tipo `json_schema` para forzar una salida estructurada: `categoria`, `sentimiento` y un `resumen` en español. Modelo `gpt-4o-mini`, suficiente para esta tarea y barato.

In [6]:
SCHEMA = {
    "name": "clasificacion_noticia",
    "schema": {
        "type": "object",
        "properties": {
            "categoria": {
                "type": "string",
                "enum": ["lesion", "transferencia", "rendimiento", "disciplinario", "personal", "otro"],
            },
            "sentimiento": {"type": "string", "enum": ["positivo", "neutral", "negativo"]},
            "resumen": {"type": "string"},
        },
        "required": ["categoria", "sentimiento", "resumen"],
        "additionalProperties": False,
    },
    "strict": True,
}


def clasificar_articulo(headline, description):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Eres un analista deportivo. Clasifica la noticia de fútbol en una "
                    "categoría, su sentimiento hacia el/los jugadores mencionados, y da un "
                    "resumen de una oración en español."
                ),
            },
            {"role": "user", "content": f"Titular: {headline}\nDescripción: {description}"},
        ],
        response_format={"type": "json_schema", "json_schema": SCHEMA},
    )
    return json.loads(resp.choices[0].message.content)

In [7]:
clasificaciones = []
for _, row in df_news.iterrows():
    resultado = clasificar_articulo(row["headline"], row["description"])
    resultado["article_id"] = row["id"]
    clasificaciones.append(resultado)

df_clasif = pd.DataFrame(clasificaciones)
df_clasif["categoria"].value_counts()

categoria
rendimiento      69
transferencia    63
otro             52
lesion           11
disciplinario     5
personal          2
Name: count, dtype: int64

In [8]:
df_clasif["sentimiento"].value_counts()

sentimiento
positivo    92
neutral     76
negativo    34
Name: count, dtype: int64

## 4. Unión jugador + clasificación

Cada fila queda como (jugador, artículo, categoría, sentimiento, resumen) — la unidad de análisis que sirve como insumo de features por jugador.

In [9]:
df_final = df_articulo_jugador.merge(df_clasif, on="article_id", how="left")
df_final.head(10)

,article_id,liga,published,headline,athlete_id,athlete_name,categoria,sentimiento,resumen
0,49905021,Premier League,2026-09-11T16:46:26Z,Monaco's Lamine Camara says he has 'turned the...,38721,Lamine Camara,transferencia,neutral,"El centrocampista del Mónaco, Lamine Camara, a..."
1,49905024,Premier League,2026-09-11T16:47:20Z,Tottenham's Roberto De Zerbi says Mykhailo Mud...,299911,Mykhailo Mudryk,lesion,negativo,"Roberto De Zerbi, entrenador del Tottenham, ca..."
2,49904487,Premier League,2026-09-11T18:35:59Z,Arsenal reach agreement to sign 17-year-old wi...,420882,Andria Bartishvili,transferencia,positivo,Arsenal ha llegado a un acuerdo para firmar al...
3,49904189,Premier League,2026-09-11T15:04:37Z,Xabi Alonso enjoys Morgan Rogers and Cole Palm...,265653,Morgan Rogers,rendimiento,positivo,Xabi Alonso destaca la amistad entre Cole Palm...
4,49904189,Premier League,2026-09-11T15:04:37Z,Xabi Alonso enjoys Morgan Rogers and Cole Palm...,296395,Cole Palmer,rendimiento,positivo,Xabi Alonso destaca la amistad entre Cole Palm...
5,49903038,Premier League,2026-09-11T12:36:27Z,Luke Shaw to face late fitness test ahead of M...,167127,Luke Shaw,lesion,neutral,Luke Shaw será evaluado para determinar si est...
6,49903001,Premier League,2026-09-11T12:27:38Z,'Kings of Manchester': City boss Enzo Maresca ...,238472,Elliot Anderson,otro,positivo,El entrenador Enzo Maresca defiende la afirmac...
7,49902523,Premier League,2026-09-11T11:37:25Z,Man United youth coach on 15-year-old starlet ...,3108507,JJ Gabriel,otro,positivo,El entrenador juvenil del Manchester United de...
8,49902002,Premier League,2026-09-11T10:36:20Z,Arteta praises 'extremely intelligent' Havertz...,231182,Kai Havertz,rendimiento,positivo,Arteta elogia la inteligencia de Havertz antes...
9,49901936,Premier League,2026-09-11T10:34:51Z,New Hull signing Sorba Thomas unhurt after fli...,250478,Sorba Thomas,otro,positivo,"El nuevo fichaje del Hull, Sorba Thomas, no re..."


## 5. Cruce contra el dataset de `valor_eur`

Normalizamos nombres (quitar acentos, minúsculas) y comparamos contra `Player` de `data/processed/fbref_tm_features.parquet` para ver qué tan aprovechable es esto como feature real del modelo.

In [10]:
df_feat = pd.read_parquet("../data/processed/fbref_tm_features.parquet")


def normalizar(nombre):
    return unidecode(str(nombre)).lower().strip()


jugadores_dataset = set(df_feat["Player"].apply(normalizar))
df_final["athlete_name_norm"] = df_final["athlete_name"].apply(normalizar)
df_final["en_dataset"] = df_final["athlete_name_norm"].isin(jugadores_dataset)

print("Match rate contra el dataset:", df_final["en_dataset"].mean().round(3))
df_final.loc[~df_final["en_dataset"], "athlete_name"].drop_duplicates()

Match rate contra el dataset: 0.86


2       Andria Bartishvili
7               JJ Gabriel
20                 Estêvão
27                   Sávio
30           Iliman Ndiaye
32          Geovany Quenda
40           Juan Iglesias
41           Enrique Salas
59             Job Ochieng
61           Fernando Niño
68      José Ángel Carmona
71        Marcos Fernández
74                 Chupete
81            Mariano Díaz
91     Zakaria Eddahchouri
95             Luismi Cruz
101             Fran Pérez
106         Robert Navarro
117           Troy Parrott
122                 Antony
145         Enzo Fernández
148                Éderson
177     Paul Okon-Engstler
204        Alexia Putellas
205         Aitana Bonmatí
206           Khadija Shaw
207         Giovanni Reyna
208            Fabián Ruiz
229           Thato Sibiya
Name: athlete_name, dtype: str

Los que no cruzan son en su mayoría esperables: jugadoras de fútbol femenino (Alexia Putellas, Aitana Bonmatí…), jugadores de filiales/cantera que no llegan al primer equipo, o transferencias/préstamos fuera de las 5 grandes ligas — no son errores de matching, son jugadores fuera del universo de `fbref_tm_features.parquet`.

## 6. Guardado

Guardamos la fotografía del día en `data/interim/` (no se sube al repo, `data/**` está en `.gitignore`) para poder ir acumulando snapshots si se corre este notebook periódicamente.

In [11]:
from datetime import date

os.makedirs("../data/interim/espn_news", exist_ok=True)
out_path = f"../data/interim/espn_news/{date.today().isoformat()}.parquet"
df_final.drop(columns=["athlete_name_norm"]).to_parquet(out_path, index=False)
print("Guardado en", out_path)

Guardado en ../data/interim/espn_news/2026-09-11.parquet


## 7. Limitaciones y próximos pasos

- **Sin histórico real**: cada corrida es un snapshot de ~3-4 días. Para usar esto como feature del modelo de `valor_eur` (que es un panel de varias temporadas) habría que correr esta captura repetidamente desde ahora y acumular — no se puede reconstruir el pasado.
- **Cobertura desigual entre jugadores**: de ~200 artículos por corrida, solo ~150 jugadores distintos quedan mencionados, y son sobre todo los de mayor perfil mediático (transferencias, lesiones de titulares). La mayoría de las ~13,000 filas del dataset de features nunca va a tener noticia asociada.
- **Costo controlado**: 1 llamado a `gpt-4o-mini` por artículo (no por jugador), del orden de ~200 llamados por corrida — costo marginal.
- **Siguiente paso sugerido**: si se decide seguir por esta vía, envolver la captura en un script que se pueda programar (cron / GitHub Action) para ir construyendo el histórico real, en vez de re-ejecutar el notebook a mano.